# Rapid Flood Extent Mapping from Sentinel-1 SAR Imagery using Multi-Temporal Attention U-Net

In [1]:

# ------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------

import os
import sys
import json
import random
import platform
import shutil
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

# ------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------

PROJECT_NAME = (
    "Rapid Flood Extent Mapping from Sentinel-1 SAR "
    "using Multi-Temporal Attention U-Net"
)

DATASET_NAME = "Kuro Siwo"
REPOSITORY = "orion-ai-lab/Kuro-Siwo-Webdataset"

SEED = 42

# Kaggle working directory
PROJECT_DIR = Path("/kaggle/working/flood_mapping_project")

# ------------------------------------------------
# 3. CREATE PROJECT DIRECTORIES
# ------------------------------------------------

DIRECTORIES = {
    "data": PROJECT_DIR / "data",
    "checkpoints": PROJECT_DIR / "checkpoints",
    "logs": PROJECT_DIR / "logs",
    "results": PROJECT_DIR / "results",
    "configs": PROJECT_DIR / "configs",
    "figures": PROJECT_DIR / "figures"
}

for directory in DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 4. SET RANDOM SEEDS
# ------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Deterministic behaviour where supported
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------------------------
# 5. SELECT COMPUTING DEVICE
# ------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------
# 6. ENVIRONMENT INFORMATION
# ------------------------------------------------

print("=" * 70)
print("PROJECT INITIALIZATION")
print("=" * 70)

print(f"Project          : {PROJECT_NAME}")
print(f"Dataset          : {DATASET_NAME}")
print(f"Repository       : {REPOSITORY}")
print(f"Python           : {platform.python_version()}")
print(f"Operating System : {platform.platform()}")
print(f"PyTorch          : {torch.__version__}")
print(f"NumPy            : {np.__version__}")
print(f"Pandas           : {pd.__version__}")

# ------------------------------------------------
# 7. HARDWARE INFORMATION
# ------------------------------------------------

print("\n" + "=" * 70)
print("HARDWARE INFORMATION")
print("=" * 70)

print(f"CPU cores        : {os.cpu_count()}")
print(f"Compute device   : {DEVICE}")

if torch.cuda.is_available():

    gpu = torch.cuda.get_device_properties(0)

    print(f"GPU              : {gpu.name}")
    print(f"GPU memory       : {gpu.total_memory / 1e9:.2f} GB")
    print(f"CUDA version     : {torch.version.cuda}")

else:
    print("GPU              : Not available")

# ------------------------------------------------
# 8. RAM INFORMATION
# ------------------------------------------------

try:
    import psutil

    memory = psutil.virtual_memory()

    print(f"Total RAM        : {memory.total / 1e9:.2f} GB")
    print(f"Available RAM    : {memory.available / 1e9:.2f} GB")

except ImportError:
    print("RAM information  : psutil not installed")

# ------------------------------------------------
# 9. STORAGE INFORMATION
# ------------------------------------------------

disk = shutil.disk_usage("/kaggle/working")

print(f"Total disk       : {disk.total / 1e9:.2f} GB")
print(f"Free disk        : {disk.free / 1e9:.2f} GB")

# ------------------------------------------------
# 10. SAVE PROJECT CONFIGURATION
# ------------------------------------------------

CONFIG = {
    "project_name": PROJECT_NAME,
    "dataset": DATASET_NAME,
    "repository": REPOSITORY,
    "random_seed": SEED,
    "device": str(DEVICE),
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "environment": "Kaggle"
}

config_path = DIRECTORIES["configs"] / "project_config.json"

with open(config_path, "w") as file:
    json.dump(CONFIG, file, indent=4)

# ------------------------------------------------
# 11. DISPLAY PROJECT STRUCTURE
# ------------------------------------------------

print("\n" + "=" * 70)
print("PROJECT DIRECTORY STRUCTURE")
print("=" * 70)

print(PROJECT_DIR)

for name, directory in DIRECTORIES.items():
    print(f"  {name:<15}: {directory}")

# ------------------------------------------------
# 12. INITIALIZATION SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("INITIALIZATION COMPLETED SUCCESSFULLY")
print("=" * 70)

print(f"Random seed      : {SEED}")
print(f"Device           : {DEVICE}")
print(f"Configuration    : {config_path}")
print(f"Project folder   : {PROJECT_DIR}")


PROJECT INITIALIZATION
Project          : Rapid Flood Extent Mapping from Sentinel-1 SAR using Multi-Temporal Attention U-Net
Dataset          : Kuro Siwo
Repository       : orion-ai-lab/Kuro-Siwo-Webdataset
Python           : 3.13.15
Operating System : Linux-6.18.55+-x86_64-with-glibc2.39
PyTorch          : 2.11.0+cu128
NumPy            : 2.1.3
Pandas           : 2.3.3

HARDWARE INFORMATION
CPU cores        : 4
Compute device   : cuda
GPU              : Tesla T4
GPU memory       : 15.64 GB
CUDA version     : 12.8
Total RAM        : 33.66 GB
Available RAM    : 31.76 GB
Total disk       : 20.96 GB
Free disk        : 20.94 GB

PROJECT DIRECTORY STRUCTURE
/kaggle/working/flood_mapping_project
  data           : /kaggle/working/flood_mapping_project/data
  checkpoints    : /kaggle/working/flood_mapping_project/checkpoints
  logs           : /kaggle/working/flood_mapping_project/logs
  results        : /kaggle/working/flood_mapping_project/results
  configs        : /kaggle/working/flood_ma

In [ ]:

# ============================================================
# CELL 2: DATASET REPOSITORY AND STORAGE AUDIT
# ============================================================

from huggingface_hub import HfApi
from huggingface_hub.hf_api import RepoFile

# ------------------------------------------------------------
# 1. Initialize Hugging Face API
# ------------------------------------------------------------

api = HfApi()

print("=" * 70)
print("KURO SIWO DATASET REPOSITORY")
print("=" * 70)

# ------------------------------------------------------------
# 2. Inspect repository shards
# ------------------------------------------------------------

SPLITS = ["train_GRD", "test_GRD"]

SHARDS = {}

for split in SPLITS:

    files = api.list_repo_tree(
        REPOSITORY,
        path_in_repo=split,
        repo_type="dataset",
        recursive=True
    )

    tar_files = [
        file for file in files
        if isinstance(file, RepoFile)
        and file.path.endswith(".tar")
    ]

    tar_files = sorted(
        tar_files,
        key=lambda file: file.path
    )

    SHARDS[split] = tar_files

    total_size = sum(file.size for file in tar_files)

    print(f"\n{split}")
    print("-" * 50)
    print(f"Number of shards : {len(tar_files)}")
    print(f"Total size       : {total_size / 1e9:.2f} GB")

    for file in tar_files:
        print(
            f"{file.path:<40} "
            f"{file.size / 1e9:>8.2f} GB"
        )

# ------------------------------------------------------------
# 3. Check existing dataset directories
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXISTING DATASET FILES")
print("=" * 70)

DATASET_LOCATIONS = [
    Path("/kaggle/working/kuro_siwo"),
    DIRECTORIES["data"]
]

for location in DATASET_LOCATIONS:

    print(f"\nLocation: {location}")

    if not location.exists():
        print("Directory does not exist.")
        continue

    tar_files = list(location.rglob("*.tar"))

    if not tar_files:
        print("No TAR files found.")
        continue

    for file in tar_files:
        print(
            f"{file} : "
            f"{file.stat().st_size / 1e9:.2f} GB"
        )

# ------------------------------------------------------------
# 4. Current storage status
# ------------------------------------------------------------

disk = shutil.disk_usage("/kaggle/working")

print("\n" + "=" * 70)
print("STORAGE STATUS")
print("=" * 70)

print(f"Total disk : {disk.total / 1e9:.2f} GB")
print(f"Used disk  : {disk.used / 1e9:.2f} GB")
print(f"Free disk  : {disk.free / 1e9:.2f} GB")

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET AUDIT COMPLETED")
print("=" * 70)

print("Repository inspection: Successful")
print("Dataset download      : Not performed")
print("Storage check         : Completed")
